# Project 5 — Module 6: Supervised Machine Learning
## Lesson 3: Preprocessing & Feature Scaling

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN |
| **Phase** | 3 — Data Preparation |
| **Module** | 6 — Supervised Machine Learning (Alkemy Bootcamp) |
| **Dataset** | PequeShop — customers_final.csv + transactions_final.csv |
| **Date** | 2026-03 |

---

> **Executive Summary:**  
> This notebook covers ML-specific data preparation for project-5.
> Data cleaning and null treatment were completed in `project-2-pequeshop-analytics`.
> New scope: encode categorical features (OneHotEncoder), scale numeric features
> (StandardScaler), assemble the final feature matrix X, and apply the
> train/test split from notebook 02 to the full preprocessed dataset.


## Table of Contents

1. [CRISP-DM Phase 3 — Data Preparation](#1-crisp-dm-phase-3--data-preparation)
2. [Environment Setup](#2-environment-setup)
3. [Load Data](#3-load-data)
4. [Null Validation — Quick Check](#4-null-validation--quick-check)
5. [Feature Engineering — Platform per Customer](#5-feature-engineering--platform-per-customer)
6. [Feature Selection — Evidence-Based](#6-feature-selection--evidence-based)
7. [Categorical Encoding — OneHotEncoder](#7-categorical-encoding--onehotencoder)
8. [Numeric Scaling — StandardScaler](#8-numeric-scaling--standardscaler)
9. [Final Feature Matrix — Assembly](#9-final-feature-matrix--assembly)
10. [Train/Test Split — Full Feature Matrix](#10-traintest-split--full-feature-matrix)
11. [LEAN Filter — Waste Elimination Review](#11-lean-filter--waste-elimination-review)
12. [Decisions Log — Lesson 3](#12-decisions-log--lesson-3)
13. [Next Steps — Lesson 4 Preview](#13-next-steps--lesson-4-preview)


---
## 1. CRISP-DM Phase 3 — Data Preparation

### 1.1 Scope — What is New vs What is Already Done

| Task | Status | Reference |
|------|--------|----------|
| Data collection and null treatment | ✅ Done | `project-2-pequeshop-analytics` |
| Outlier analysis | ✅ Done | `project-3-eda-pequeshop` |
| Feature validation (H1–H4) | ✅ Done | `project-4b-pequeshop-statistical-inference` |
| **Categorical encoding (OneHotEncoder)** | 🔲 New — this notebook | ML requirement |
| **Numeric scaling (StandardScaler)** | 🔲 New — this notebook | ML requirement |
| **Final feature matrix assembly** | 🔲 New — this notebook | ML requirement |
| **Train/test split on full matrix** | 🔲 New — this notebook | ML requirement |

> **Lean rule:** Only the four tasks marked 🔲 are performed here.


In [1]:
# ===== Environment Setup =====
import warnings
warnings.formatwarning = lambda msg, *args, **kwargs: f'Warning: {msg}\n'
warnings.simplefilter('always')
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# ===== Plot Style =====
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')

# ===== Reproducibility =====
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ===== Paths =====
DATA_PROCESSED  = Path('../data/processed')
DATA_FINAL      = Path('../data/final')
REPORTS_FIGURES = Path('../reports/figures')
MODELS_PATH     = Path('../models')
DATA_FINAL.mkdir(parents=True, exist_ok=True)
REPORTS_FIGURES.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

print('Environment ready.')
print(f'Data path   : {DATA_PROCESSED}')
print(f'Figures path: {REPORTS_FIGURES}')

Environment ready.
Data path   : ..\data\processed
Figures path: ..\reports\figures


---
## 3. Load Data


In [2]:
# ===== Load Datasets =====
df_customers    = pd.read_csv(DATA_PROCESSED / 'customers_final.csv')
df_transactions = pd.read_csv(DATA_PROCESSED / 'transactions_final.csv')

print(f'customers    : {df_customers.shape}')
print(f'transactions : {df_transactions.shape}')

customers    : (500, 16)
transactions : (1192, 19)


---
## 4. Null Validation — Quick Check

> Data was cleaned in `project-2-pequeshop-analytics`.
> This is a quick validation only — not a full treatment.


In [3]:
# ===== Null Validation =====
nulls = df_customers.isnull().sum()
nulls = nulls[nulls > 0]

if len(nulls) == 0:
    print('✅ No nulls in customers_final.csv — data is clean.')
else:
    print('⚠️  Nulls found:')
    print(nulls)

nulls_t = df_transactions.isnull().sum()
nulls_t = nulls_t[nulls_t > 0]

if len(nulls_t) == 0:
    print('✅ No nulls in transactions_final.csv — data is clean.')
else:
    print('⚠️  Nulls found in transactions:')
    print(nulls_t)

⚠️  Nulls found:
nps_score                   265
total_transactions          108
total_revenue               108
avg_ticket                  108
first_purchase              108
last_purchase               108
days_since_last_purchase    108
tenure_days                 108
retargeting_segment         108
dtype: int64
✅ No nulls in transactions_final.csv — data is clean.


---
## 5. Feature Engineering — Platform per Customer


In [4]:
# ===== Primary Platform per Customer =====
# Assign dominant platform by mode (same logic as project-4b H2)
platform_map = (
    df_transactions.groupby('customer_id')['platform']
    .agg(lambda x: x.mode()[0])
    .reset_index()
    .rename(columns={'platform': 'primary_platform'})
)
df_merged = df_customers.merge(platform_map, on='customer_id', how='left')

print('Platform distribution:')
print(df_merged['primary_platform'].value_counts())

Platform distribution:
primary_platform
mercadolibre    297
shopify          95
Name: count, dtype: int64


---
## 6. Feature Selection — Evidence-Based

Features selected based on `project-4b-pequeshop-statistical-inference` results:

| Feature | Type | Evidence | Include? |
|---------|------|----------|----------|
| `primary_platform` | Categorical | H2 rejected (p=0.024) | ✅ Yes |
| `retargeting_segment` | Categorical | H3 rejected (p<0.001) | ✅ Yes |
| `total_transactions` | Numeric | Transaction volume proxy | ✅ Yes |
| `total_revenue` | Numeric | Correlated with ticket | ✅ Yes |
| `nps_category` | Categorical | H4 not rejected (p=0.780) | ❌ Excluded |


In [5]:
# ===== Define Features =====
CATEGORICAL_FEATURES = ['primary_platform', 'retargeting_segment']
NUMERIC_FEATURES     = ['total_transactions', 'total_revenue']
TARGET               = 'avg_ticket'

# Assemble base dataframe
df_model = df_merged[CATEGORICAL_FEATURES + NUMERIC_FEATURES + [TARGET]].dropna()

print(f'Feature matrix shape: {df_model.shape}')
print(f'Categorical: {CATEGORICAL_FEATURES}')
print(f'Numeric    : {NUMERIC_FEATURES}')
print(f'Target     : {TARGET}')
print()
print(df_model.head())

Feature matrix shape: (392, 5)
Categorical: ['primary_platform', 'retargeting_segment']
Numeric    : ['total_transactions', 'total_revenue']
Target     : avg_ticket

  primary_platform retargeting_segment  total_transactions  total_revenue  \
0     mercadolibre              Active                 9.0       268360.0   
1     mercadolibre             At Risk                 5.0       164575.0   
2     mercadolibre              Active                12.0       283010.0   
3     mercadolibre             Dormant                 3.0        82302.0   
4     mercadolibre             At Risk                 7.0       200509.0   

     avg_ticket  
0  29817.777778  
1  32915.000000  
2  23584.166667  
3  27434.000000  
4  28644.142857  


---
## 7. Categorical Encoding — OneHotEncoder

Categorical features must be encoded before ML models can process them.
OneHotEncoder creates binary columns for each category — no ordinal
assumption is imposed.

| Feature | Categories | Encoding |
|---------|-----------|----------|
| `primary_platform` | mercadolibre, shopify | 1 binary column (drop first) |
| `retargeting_segment` | Active, At Risk, Dormant | 2 binary columns (drop first) |


In [6]:
# ===== Preprocessing Pipeline =====
categorical_transformer = OneHotEncoder(drop='first', sparse_output=False)
numeric_transformer     = StandardScaler()

preprocessor = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, CATEGORICAL_FEATURES),
    ('num', numeric_transformer,     NUMERIC_FEATURES)
])

X = df_model[CATEGORICAL_FEATURES + NUMERIC_FEATURES]
y = df_model[TARGET]

# Fit on full data to inspect — actual fit will be on X_train only
X_transformed = preprocessor.fit_transform(X)

# Get feature names
cat_names = preprocessor.named_transformers_['cat'].get_feature_names_out(CATEGORICAL_FEATURES)
feature_names = list(cat_names) + NUMERIC_FEATURES

df_transformed = pd.DataFrame(X_transformed, columns=feature_names)

print('Encoded feature matrix:')
print(f'Shape: {df_transformed.shape}')
print(f'Features: {feature_names}')
print()
display(df_transformed.head())

Encoded feature matrix:
Shape: (392, 5)
Features: ['primary_platform_shopify', 'retargeting_segment_At Risk', 'retargeting_segment_Dormant', 'total_transactions', 'total_revenue']



,primary_platform_shopify,retargeting_segment_At Risk,retargeting_segment_Dormant,total_transactions,total_revenue
0,0.0,0.0,0.0,2.526198,2.008156
1,0.0,1.0,0.0,0.830531,0.800492
2,0.0,0.0,0.0,3.797949,2.178626
3,0.0,0.0,1.0,-0.017303,-0.156854
4,0.0,1.0,0.0,1.678365,1.218628


---
## 8. Numeric Scaling — StandardScaler

StandardScaler transforms numeric features to mean=0, std=1.
This is required for distance-based models (KNN) and regularized
models (Ridge, Lasso) to avoid dominance by high-magnitude features.

| Feature | Before Scaling | After Scaling |
|---------|---------------|---------------|
| `total_transactions` | Raw count | mean=0, std=1 |
| `total_revenue` | CLP amount (large values) | mean=0, std=1 |

> **Note:** Scaling is applied **inside the pipeline** — fit on training
> data only, then transform both train and test. This prevents data leakage.


In [7]:
# ===== Scaling Verification =====
num_cols = [c for c in feature_names if c in NUMERIC_FEATURES]
print('Scaled numeric features — verification (mean ≈ 0, std ≈ 1):')
print(df_transformed[num_cols].describe().round(4))

Scaled numeric features — verification (mean ≈ 0, std ≈ 1):
       total_transactions  total_revenue
count            392.0000       392.0000
mean               0.0000         0.0000
std                1.0013         1.0013
min               -0.8651        -1.0332
25%               -0.8651        -0.7316
50%               -0.4412        -0.3374
75%                0.4066         0.3898
max                4.2219         3.5744


---
## 9. Final Feature Matrix — Assembly


In [8]:
# ===== Final Feature Matrix Summary =====
print('FINAL FEATURE MATRIX')
print('=' * 45)
print(f'Samples  : {X_transformed.shape[0]}')
print(f'Features : {X_transformed.shape[1]}')
print()
for i, name in enumerate(feature_names):
    ftype = 'Encoded categorical' if name in cat_names else 'Scaled numeric'
    print(f'  {i+1}. {name:<35} [{ftype}]')
print()
print(f'Target   : {TARGET} (not scaled — regression target)')

# Save preprocessed data to final/
df_final = df_transformed.copy()
df_final[TARGET] = y.values
df_final.to_csv(DATA_FINAL / 'features_final.csv', index=False)
print()
print(f'✅ Saved: {DATA_FINAL}/features_final.csv')

FINAL FEATURE MATRIX
Samples  : 392
Features : 5

  1. primary_platform_shopify            [Encoded categorical]
  2. retargeting_segment_At Risk         [Encoded categorical]
  3. retargeting_segment_Dormant         [Encoded categorical]
  4. total_transactions                  [Scaled numeric]
  5. total_revenue                       [Scaled numeric]

Target   : avg_ticket (not scaled — regression target)

✅ Saved: ..\data\final/features_final.csv


---
## 10. Train/Test Split — Full Feature Matrix


In [9]:
# ===== Train/Test Split on Full Feature Matrix =====
# Preprocessor will be fit on X_train only inside the modeling pipeline
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print('TRAIN/TEST SPLIT — Full Feature Matrix')
print('=' * 45)
print(f'Total   : {len(X)}')
print(f'Train   : {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Test    : {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)')
print()
print('X_train shape:', X_train.shape)
print('X_test  shape:', X_test.shape)

# Save preprocessor for reuse in notebooks 04-06
joblib.dump(preprocessor, MODELS_PATH / 'preprocessor_v1.pkl')
print()
print(f'✅ Preprocessor saved: {MODELS_PATH}/preprocessor_v1.pkl')
print('   X_train, X_test, y_train, y_test ready for notebook 04.')

TRAIN/TEST SPLIT — Full Feature Matrix
Total   : 392
Train   : 313 (80%)
Test    : 79 (20%)

X_train shape: (313, 4)
X_test  shape: (79, 4)

✅ Preprocessor saved: ..\models/preprocessor_v1.pkl
   X_train, X_test, y_train, y_test ready for notebook 04.


---
## 11. LEAN Filter — Waste Elimination Review

| LEAN Question | Answer | Action |
|---------------|--------|--------|
| Did we repeat null treatment from project-2? | ✅ No — quick validation only | No waste |
| Did we repeat outlier analysis from project-3? | ✅ No — referenced only | No waste |
| Is OneHotEncoder the right choice for nominal categories? | ✅ Yes — no ordinal assumption imposed | Keep |
| Why drop='first' in OneHotEncoder? | ✅ Avoids multicollinearity (dummy variable trap) | Correct |
| Is StandardScaler necessary for all models? | ✅ Required for KNN, Ridge, Lasso — harmless for Linear Regression | Apply universally |
| Is the preprocessor saved as .pkl? | ✅ Yes — reused in notebooks 04-06, no re-fitting | MLOps standard |


---
## 12. Decisions Log — Lesson 3

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|----------|-----------|------------------------|-------------|
| 1 | OneHotEncoder with drop='first' | Avoids dummy variable trap (multicollinearity) in linear models | LabelEncoder (imposes ordinal assumption) | ✅ Statistically correct |
| 2 | StandardScaler for numeric features | Required for KNN and Ridge/Lasso; consistent preprocessing across all models | MinMaxScaler (sensitive to outliers) | ✅ More robust choice |
| 3 | Exclude nps_category from features | H4 not rejected (F=0.25, p=0.780) — no predictive value confirmed statistically | Include NPS and let model decide | ✅ Evidence-based, reduces noise |
| 4 | Use ColumnTransformer pipeline | Ensures encoding and scaling are applied correctly to each feature type; prevents data leakage | Manual encoding then scaling | ✅ MLOps standard |
| 5 | Save preprocessor as .pkl | Reused in notebooks 04-06 without re-fitting — avoids leakage and ensures consistency | Re-fit preprocessor in each notebook | ✅ No re-work |
| 6 | Fit preprocessor on full X for inspection, split before modeling | Visual inspection of encoded matrix; actual modeling uses train-only fit via pipeline | Fit only on X_train from start | ✅ Transparency + correctness |


---
## 13. Next Steps — Lesson 4 Preview

In **Lesson 4 — Modeling (notebook 04)**, the following tasks will be completed:

1. Load preprocessor from `models/preprocessor_v1.pkl`
2. Build ML pipeline: preprocessor → model
3. Train **Linear Regression** baseline
4. Train **Polynomial Regression** (degree=2)
5. Train **KNN Regressor** (contrast with classification concept)
6. Compare train vs test performance across models
7. Interpret Linear Regression coefficients

---

**Previous:** [02 - Data Understanding](./02_data_understanding.ipynb)  
**Next Phase -->** [04 - Modeling](./04_modeling.ipynb)

*Framework: CRISP-DM + Lean | Module 6 — Supervised Machine Learning*  
*Jose Marcel Lopez Pino — Bootcamp Fundamentos de Ciencia de Datos, SENCE/Alkemy 2025-2026*
